In [0]:
import sys
sys.path.append("/Workspace/Repos/juliano.garciagregorio@gmail.com/mentoria-dados/data-product-crypto/src")
import zordon

In [0]:
import requests
import time
from datetime import datetime, timezone

import zordon

proj = zordon.Project(spark, country="br", region="sa", environment="dev")

bronze_poloniex = proj.client(layer="bronze", domain="poloniex", subdomain="ohlcv")

In [0]:
def fetch_poloniex_candles(symbol, interval, start_ts=None, end_ts=None, limit=500):
    url = f"https://api.poloniex.com/markets/{symbol}/candles"
    params = {"interval": interval, "limit": limit}
    if start_ts:
        params["startTime"] = start_ts
    if end_ts:
        params["endTime"] = end_ts

    for tentativa in range(5):
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code == 200:
            return resp.json()
        elif resp.status_code == 429:
            time.sleep(2 ** tentativa)
        else:
            resp.raise_for_status()
    raise Exception("Falha após múltiplas tentativas de fetch (Poloniex)")

In [0]:
symbol = "BTC_USDT"
interval = "HOUR_1"

try:
    df_existente = bronze_poloniex.read_table(
        "candles",
        filters={"symbol": symbol, "interval": interval}
    )
    last_start_time = df_existente.agg({"start_time": "max"}).collect()[0][0]
    start_ts = int(last_start_time) + 1 if last_start_time else None
except Exception:
    start_ts = None

In [0]:
from datetime import datetime, timezone

raw_data = fetch_poloniex_candles(symbol, interval, start_ts=start_ts)
ingestion_ts = datetime.now(timezone.utc).isoformat()

if not raw_data:
    print(f"[{ingestion_ts}] Nenhum candle novo para {symbol}/{interval} (start_ts={start_ts}).")
else:
    registros = [
        {
            "symbol": symbol,
            "interval_requested": interval,
            "low": candle[0],
            "high": candle[1],
            "open": candle[2],
            "close": candle[3],
            "amount": candle[4],
            "quantity": candle[5],
            "buy_taker_amount": candle[6],
            "buy_taker_quantity": candle[7],
            "trade_count": candle[8],
            "ts": candle[9],
            "weighted_average": candle[10],
            "interval": candle[11],
            "start_time": candle[12],
            "close_time": candle[13],
            "ingestion_timestamp": ingestion_ts
        }
        for candle in raw_data
    ]

    df_novo = spark.createDataFrame(registros)

    fqn = bronze_poloniex.upsert_table(
        df_novo,
        "candles",
        merge_keys=["symbol", "interval", "start_time"],
        partition_cols=["symbol"]
    )
    print(f"[{ingestion_ts}] {len(registros)} candles processados em {fqn}")